# Semana 4 · Sesión 2: Reescribir, sustituir y evaluar

**Módulo 1**

## Objetivos de la sesión

1. Reescribir una expresión a voluntad con `expand`, `factor`, `collect`,
   `trigsimp` y `radsimp`, y saber cuándo conviene `simplify`.
2. Sustituir valores y expresiones con `subs`, distinguiendo la
   sustitución encadenada de la simultánea.
3. Bajar del símbolo al número con `evalf` y `lambdify`, y graficar el
   resultado.

## Retomamos

En la sesión 1 vimos que una expresión de SymPy es un árbol que se
construye solo hasta donde sale barato: `x + x` se junta en `2*x`, pero
`(x + 1)**2` se queda sin expandir. Hoy tomamos el control de esa segunda
parte.

Este notebook es autocontenido: la celda de abajo vuelve a importar y a
declarar todo lo que necesita. Hoy sí aparecen NumPy y Matplotlib —los de
la semana 1— porque al final vamos a graficar.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

sp.init_printing()

# Símbolos del día. Las suposiciones son las del problema físico:
# velocidad inicial, gravedad y constantes elásticas son positivas;
# posición, tiempo y ángulo son reales.
x, t, theta = sp.symbols("x t theta", real=True)
v0, g, k1, k2 = sp.symbols("v0 g k1 k2", positive=True)

## La familia de reescritura

Todas estas funciones reciben una expresión y devuelven **otra expresión**
matemáticamente equivalente, escrita de otra forma. Ninguna modifica la
original: los objetos de SymPy son inmutables, como los de la semana 2.

| Función | Qué hace | Cuándo la quieres |
|---|---|---|
| `sp.expand` | Multiplica y distribuye | Abrir paréntesis, comparar término a término |
| `sp.factor` | Escribe como producto de factores | Encontrar raíces, ver ceros |
| `sp.collect` | Agrupa por potencias de un símbolo | Leer coeficientes, órdenes de un parámetro |
| `sp.cancel` | Reduce una fracción racional | Quitar factores comunes |
| `sp.trigsimp` | Aplica identidades trigonométricas | Cerrar $\sin^2 + \cos^2$ |
| `sp.radsimp` | Racionaliza denominadores con raíces | Dejar el denominador limpio |
| `sp.simplify` | Prueba de todo y se queda con lo "más simple" | Cuando no sabes qué quieres |

Las seis primeras son **dirigidas**: le dices exactamente qué hacer.
`simplify` es la excepción, y por eso la dejamos para el final.

In [ ]:
expresion = (x + 1)**2

print("original :", expresion)
print("expand   :", sp.expand(expresion))
print("factor   :", sp.factor(sp.expand(expresion)))

# La original sigue intacta: expand devolvió una expresión nueva.
expresion

## `collect`: agrupar por potencias

`collect` es la que menos se conoce y la que más se usa en física. Agrupa
los términos por potencias de un símbolo y saca los coeficientes a la
vista.

Ejemplo: la energía potencial de una masa unida a **dos resortes en
paralelo**, con constantes $k_1$ y $k_2$, y una fuerza externa constante
$F$:

$$V(x) = \tfrac{1}{2}k_1 x^2 + \tfrac{1}{2}k_2 x^2 + F x$$

Agrupada por potencias de $x$, aparece sola la constante elástica efectiva
$k_1 + k_2$ — que es justamente el resultado físico que uno quiere leer.

In [ ]:
F = sp.Symbol("F", real=True)

potencial = sp.Rational(1, 2)*k1*x**2 + sp.Rational(1, 2)*k2*x**2 + F*x

print("sin agrupar:", potencial)
print("agrupada   :", sp.collect(potencial, x))

## `trigsimp` y `radsimp`

`trigsimp` aplica identidades trigonométricas. Es la que cierra el alcance
del tiro parabólico, que es el ejemplo que vamos a arrastrar el resto de la
sesión:

$$R = \frac{2 v_0^2 \sin\theta \cos\theta}{g}
\;\xrightarrow{\;\text{trigsimp}\;}\;
\frac{v_0^2 \sin 2\theta}{g}$$

`radsimp` racionaliza denominadores: quita las raíces de abajo,
multiplicando por el conjugado.

In [ ]:
alcance = 2*v0**2*sp.sin(theta)*sp.cos(theta) / g

print("alcance          :", alcance)
print("alcance trigsimp :", sp.trigsimp(alcance))

# radsimp: la raíz sale del denominador
print("radsimp          :", sp.radsimp(1 / (sp.sqrt(2) + 1)))

## `simplify`: la navaja suiza, y por qué no abusar

`sp.simplify` prueba muchas transformaciones y se queda con la que le
parece "más simple" según una medida interna. Es cómoda, pero tiene tres
inconvenientes que conviene tener presentes:

1. Es **lenta**: en expresiones grandes puede tardar muchísimo.
2. No está **garantizada**: puede no encontrar la forma que buscas.
3. "Más simple" es su criterio, no el tuyo — a veces devuelve algo distinto
   de lo que un físico llamaría la forma canónica.

Úsala cuando explores o cuando no sepas qué transformación pedir. Cuando sí
lo sepas, la función dirigida es mejor.

Donde `simplify` sí es la herramienta correcta es en la pregunta de la
sesión 1: **¿son iguales estas dos expresiones?** Ahí no te importa la
forma del resultado, solo si la diferencia es cero.

In [ ]:
identidad = sp.sin(x)**2 + sp.cos(x)**2

print("simplify:", sp.simplify(identidad))

# El uso más valioso: decidir igualdad matemática (sesión 1).
a = (x + 1)**2
b = x**2 + 2*x + 1
print("¿a y b son la misma?", sp.simplify(a - b) == 0)

## TODO en clase 1

La energía total de un oscilador armónico, escrita como suma de energía
cinética y potencial, es

$$E = \tfrac{1}{2} m \omega^2 A^2 \sin^2(\omega t)
    + \tfrac{1}{2} m \omega^2 A^2 \cos^2(\omega t)$$

1. Declara `m`, `omega` y `amplitud` como positivos (`tiempo` ya lo tienes
   como `t`).
2. Escribe la expresión, usando `sp.Rational(1, 2)` para la fracción.
3. Ciérrala con `sp.trigsimp` y comprueba que queda
   $\tfrac{1}{2} m \omega^2 A^2$ — es decir, que la energía **no depende
   del tiempo**, que es justo lo que dice la conservación de la energía.
4. Prueba también con `sp.simplify` y compara qué tan distinto es el
   camino para llegar al mismo lugar.

In [ ]:
# TODO en clase: escribe la energía del oscilador y ciérrala con trigsimp
m = ...
omega = ...
amplitud = ...

energia_total = ...

## `subs`: cambiar una cosa por otra

`subs` sustituye dentro de una expresión y devuelve **una nueva**. Puede
sustituir un símbolo por un número, por otro símbolo o por una expresión
completa — y esa última es la que más juego da en física: sustituir una
variable por su ley de movimiento cambia de qué depende la fórmula.

Igual que todo en SymPy, no modifica la original.

In [ ]:
trayectoria = x*sp.tan(theta) - g*x**2 / (2*v0**2*sp.cos(theta)**2)

print("por un número   :", trayectoria.subs(theta, sp.pi/4))

# Por una expresión completa: si la posición horizontal avanza como
# x = v0*cos(theta)*t, sustituirla convierte y(x) en y(t).
altura_en_el_tiempo = trayectoria.subs(x, v0*sp.cos(theta)*t)
print("por una expresión:", sp.simplify(altura_en_el_tiempo))

# Varias a la vez, con un diccionario:
print("varias a la vez  :", alcance.subs({v0: 20, g: sp.Rational(981, 100)}))

trayectoria

## Depuración en vivo: encadenada vs. simultánea

Cuando sustituyes varias cosas a la vez, SymPy las aplica **una tras
otra**, y el resultado de la primera entra a la segunda. Casi siempre da
igual, pero cuando las sustituciones se pisan entre sí el resultado
sorprende.

Predice qué imprime la celda antes de ejecutarla: partimos de $a + b$,
cambiamos $a$ por $b$ y luego $b$ por $0$.

In [ ]:
a_sim, b_sim = sp.symbols("a b")
suma = a_sim + b_sim

# Encadenada (el comportamiento por defecto): a -> b lo convierte en b + b,
# y entonces b -> 0 se lleva TODO por delante.
print("encadenada:", suma.subs([(a_sim, b_sim), (b_sim, 0)]))

# Simultánea: las dos sustituciones miran la expresión original.
print("simultánea:", suma.subs([(a_sim, b_sim), (b_sim, 0)], simultaneous=True))

La encadenada da `0` y la simultánea da `b`. Ninguna está mal: son dos
operaciones distintas, y la que quieres depende del problema.

**Regla práctica:** si los símbolos que sustituyes aparecen también en el
lado derecho de otra sustitución (cambios de variable, intercambios), usa
`simultaneous=True`. Si solo metes números, da igual.

## TODO en clase 2

Toma la `trayectoria` del tiro parabólico de arriba y evalúala para un
lanzamiento concreto: $v_0 = 20\ \mathrm{m/s}$, $\theta = \pi/4$ y
$g = 9.81\ \mathrm{m/s^2}$.

1. Sustituye los tres valores de una sola vez, con un diccionario. Usa
   `sp.Rational(981, 100)` para la gravedad, no `9.81`, y explica por qué
   (pista: sesión 1, los flotantes se contagian).
2. Simplifica el resultado. Debe quedarte una expresión que solo dependa de
   $x$.

In [ ]:
# TODO en clase: sustituye los tres valores en la trayectoria
valores = ...

trayectoria_concreta = ...

## Del símbolo al número: `evalf`

`evalf` (o su alias `sp.N`) calcula el valor numérico de una expresión
**con la precisión que le pidas**, no con los 15 dígitos de un flotante.
Esa es su gracia: SymPy trabaja con precisión arbitraria.

Requiere que ya no queden símbolos libres: primero `subs`, después `evalf`.

In [ ]:
print(sp.pi.evalf())      # precisión por defecto: 15 dígitos
print(sp.pi.evalf(40))    # 40 dígitos, si los necesitas

# El alcance para v0 = 20 m/s, theta = pi/4, g = 9.81 m/s^2
alcance_concreto = alcance.subs({v0: 20, theta: sp.pi/4, g: sp.Rational(981, 100)})

print("exacto  :", sp.trigsimp(alcance_concreto))
print("numérico:", alcance_concreto.evalf(6), "m")

## `lambdify`: cuando necesitas miles de evaluaciones

`evalf` está bien para un valor. Para graficar necesitas cientos o miles, y
hacerlo con `subs` + `evalf` en un bucle es lentísimo: cada llamada
reconstruye el árbol entero.

`sp.lambdify` compila la expresión a una **función de Python normal** que
opera sobre arreglos de NumPy. Es el puente entre el mundo simbólico y el
numérico de la semana 1.

Una advertencia importante: al pasar por `lambdify` **sales** del mundo
simbólico. Lo que sale del otro lado son flotantes, sin suposiciones ni
exactitud. Por eso conviene hacer todo el álgebra primero y lambdificar al
final.

In [ ]:
# Primero todo el álgebra en símbolos...
trayectoria_45 = trayectoria.subs(
    {v0: 20, theta: sp.pi/4, g: sp.Rational(981, 100)}
)

# ...y solo al final bajamos a números.
altura = sp.lambdify(x, trayectoria_45, "numpy")

posiciones = np.linspace(0, 41, 5)
altura(posiciones)   # una llamada, todo el arreglo

In [ ]:
posiciones = np.linspace(0, 40.8, 300)

fig, ax = plt.subplots()
ax.plot(posiciones, altura(posiciones))
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_title(r"Tiro parabólico: $v_0 = 20$ m/s, $\theta = \pi/4$")
plt.show()

## Las tres formas de dar números, comparadas

| Herramienta | Devuelve | Para | Costo |
|---|---|---|---|
| `subs` | una expresión de SymPy | Fijar un parámetro y seguir en simbólico | Barato una vez, caro en bucle |
| `evalf` | un número de SymPy, con la precisión que pidas | Un valor concreto, con muchos dígitos | Medio |
| `lambdify` | una función de Python/NumPy | Miles de evaluaciones, graficar | Caro una vez, luego rapidísimo |

El orden natural es siempre el mismo: **álgebra en simbólico, números al
final**.

## TODO en clase 3

Compara el alcance del tiro parabólico para varios ángulos y confirma
gráficamente que el máximo está en $45°$.

1. Completa `trayectoria_para(angulo)`: debe sustituir en `trayectoria` la
   $v_0$, la $g$ y el ángulo que recibe, y devolver la función de NumPy que
   sale de `lambdify`.
2. Descomenta las últimas líneas y grafica las tres trayectorias en los
   mismos ejes.
3. Mira la gráfica: además del máximo en $45°$, hay algo más que salta a la
   vista sobre los tiros de $30°$ y $60°$. Explícalo con la fórmula del
   alcance que cerramos con `trigsimp` al principio de la sesión.

Nota: el ángulo entra en radianes. `sp.rad(30)` convierte grados a
radianes de forma exacta, sin contaminar con flotantes.

In [ ]:
# TODO en clase: completa la función y descomenta el bloque de la gráfica
def trayectoria_para(angulo):
    """Devuelve la altura y(x) como función de NumPy, para el ángulo dado."""
    ...


# fig, ax = plt.subplots()
# for grados in (30, 45, 60):
#     altura_angulo = trayectoria_para(sp.rad(grados))
#     ax.plot(posiciones, altura_angulo(posiciones), label=f"{grados}°")
# ax.axhline(0, color="gray", linewidth=0.8)
# ax.set_ylim(0, 25)
# ax.set_xlabel("x [m]")
# ax.set_ylabel("y [m]")
# ax.legend()
# plt.show()

## Resumen

Hoy tomamos el control de la reescritura: `expand` y `factor` para abrir y
cerrar, `collect` para leer coeficientes (y de paso descubrir que dos
resortes en paralelo suman sus constantes), `trigsimp` y `radsimp` para lo
que su nombre dice, y `simplify` como último recurso — cómodo, lento y sin
garantías, pero imbatible para decidir si dos expresiones son la misma.

Después vimos `subs`, con su trampa de la sustitución encadenada, y el
camino de bajada al número: `evalf` para un valor con la precisión que
quieras, `lambdify` para miles de ellos y para graficar. La regla que
resume la sesión: **álgebra en simbólico, números al final**.

**Tarea de esta semana:** [`tarea-04.ipynb`](../tarea/tarea-04.ipynb), que
se entrega por Pull Request dentro de tu fork, como la de la semana 3 (ver
[`docs/git-guia.md`](../../docs/git-guia.md)).

**Próxima clase — Semana 5:** cálculo simbólico. Derivadas, integrales,
límites y series — y con `dsolve`, ecuaciones diferenciales. Todo lo de hoy
se vuelve la base para manipular los resultados que salgan de ahí.